In [1]:
!unzip /content/REES.zip

Archive:  /content/REES.zip
  inflating: zzzbruh/resume_classic.pdf  
  inflating: zzzbruh/resume_minimalist.pdf  
  inflating: zzzbruh/resume_modern_sections.pdf  
  inflating: zzzbruh/resume_table_multiline.pdf  
  inflating: zzzbruh/resume_two_column.pdf  
  inflating: zzzbruh/Sarvam_Jha_Artificial Intelligence.pdf  
  inflating: zzzbruh/yodolegend.pdf  


In [1]:
# ===================================
# INSTALL DEPENDENCIES
# ===================================
!pip install fastapi uvicorn pyngrok python-multipart pymupdf sentence-transformers spacy nest_asyncio -q
!python -m spacy download en_core_web_sm -q

# ===================================
# IMPORTS
# ===================================
import os
import re
import string
import tempfile
import nest_asyncio
import threading

import fitz
import spacy
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import JSONResponse
from sentence_transformers import SentenceTransformer, util
from pyngrok import ngrok
import uvicorn

# ===================================
# 🔐 SET YOUR NGROK TOKEN
# ===================================
ngrok.set_auth_token("3ARTsKaxgo1egIfA8NONMzYFVsr_6mDS7cVa3uLKeYema3vgC")

# Kill old tunnels (important)
ngrok.kill()

# ===================================
# LOAD MODELS
# ===================================
nlp = spacy.load("en_core_web_sm")
model = SentenceTransformer("all-MiniLM-L6-v2")

# ===================================
# CREATE APP
# ===================================
app = FastAPI(title="AI Resume Screening API")

# ===================================
# HELPER FUNCTIONS
# ===================================
def extract_resume_text(pdf_path):
    text = ""
    doc = fitz.open(pdf_path)

    for page in doc:
        text += page.get_text()

    return text


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    doc = nlp(text)
    tokens = [t.lemma_ for t in doc if not t.is_stop]
    return " ".join(tokens)


def keyphrase_score(text, skills):
    hits = sum(1 for s in skills if s.lower() in text)
    return hits / len(skills) if skills else 0


def semantic_score(resume_text, job_description):
    emb_resume = model.encode(resume_text, convert_to_tensor=True)
    emb_job = model.encode(job_description, convert_to_tensor=True)
    sim = util.pytorch_cos_sim(emb_resume, emb_job)
    return float(sim)


# ===================================
# ROUTES
# ===================================
@app.get("/")
def home():
    return {"message": "Resume API is running 🚀"}


@app.post("/analyze")
async def analyze_resume(
    job_description: str = Form(...),
    skills: str = Form(...),
    file: UploadFile = File(...)   # ✅ SINGLE FILE
):

    skills_list = [s.strip() for s in skills.split(",")]

    with tempfile.TemporaryDirectory() as temp_dir:
        file_path = os.path.join(temp_dir, file.filename)

        with open(file_path, "wb") as f:
            f.write(await file.read())

        raw_text = extract_resume_text(file_path)
        clean_resume = preprocess_text(raw_text)
        clean_job = preprocess_text(job_description)

        k = keyphrase_score(clean_resume, skills_list)
        s = semantic_score(clean_resume, clean_job)
        final = 0.4 * k + 0.6 * s

    return JSONResponse(content={
        "filename": file.filename,
        "final_score": round(final, 3),
        "semantic_score": round(s, 3),
        "keyphrase_score": round(k, 3)
    })


# ===================================
# RUN SERVER + NGROK
# ===================================
nest_asyncio.apply()

def run():
    uvicorn.run(app, host="0.0.0.0", port=8001)

thread = threading.Thread(target=run)
thread.start()

public_url = ngrok.connect(8001)

print("\n🚀 Your API URL:", public_url)
print("📄 Swagger Docs:", str(public_url) + "/docs")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

INFO:     Started server process [849]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)



🚀 Your API URL: NgrokTunnel: "https://pseudoinvalidly-shrewlike-claud.ngrok-free.dev" -> "http://localhost:8001"
📄 Swagger Docs: NgrokTunnel: "https://pseudoinvalidly-shrewlike-claud.ngrok-free.dev" -> "http://localhost:8001"/docs
